In [5]:
import geopandas as gpd
import xarray as xr
import pandas as pd
from rasterstats import zonal_stats
import os
import rioxarray
import rasterio

# 1. File Paths (Update these to match your files)
shapefile_path = "glaciers_for_gee.shp"
netcdf_path = r"C:\Users\rubel\Downloads\ITS_LIVE_velocity_120m_RGI14A_0000_V02.1.nc"
output_csv = "glacier_velocities_5_epochs.csv"

cog_urls = {
    1990: r"C:\Users\rubel\Downloads\ITS_LIVE_velocity_120m_RGI14A_1990_V02.1_v.tif",
    2000: r"C:\Users\rubel\Downloads\ITS_LIVE_velocity_120m_RGI14A_2000_V02.1_v.tif",
    2010: r"C:\Users\rubel\Downloads\ITS_LIVE_velocity_120m_RGI14A_2010_V02.1_v.tif",
    2015: r"C:\Users\rubel\Downloads\ITS_LIVE_velocity_120m_RGI14A_2015_V02.1_v.tif",
    2020: r"C:\Users\rubel\Downloads\ITS_LIVE_velocity_120m_RGI14A_2020_V02.1_v.tif"
}

# 2. Load Glacier Shapefile
print("Loading glacier geometries...")
gdf = gpd.read_file(shapefile_path)

if 'RGIId' not in gdf.columns:
    raise ValueError("The shapefile does not contain an 'RGIId' column!")

results_df = pd.DataFrame({'RGIId': gdf['RGIId']})

# 3. Stream Data from COGs
for year, url in cog_urls.items():
    print(f"\nStreaming data for {year}...")
    
    try:
        # Open the remote COG just to read its metadata/CRS
        with rasterio.open(url) as src:
            cog_crs = src.crs
            
            # Align CRS if they don't match
            if gdf.crs != cog_crs:
                print("  -> Reprojecting shapefile to match remote COG...")
                gdf_projected = gdf.to_crs(cog_crs)
            else:
                gdf_projected = gdf

            print("  -> Calculating zonal statistics over the cloud...")
            # rasterstats natively supports reading directly from remote URLs!
            stats = zonal_stats(
                gdf_projected, 
                url,               # Point directly to the AWS URL
                stats="mean", 
                nodata=-32767.0
            )
            
            results_df[f'v_mean_{year}'] = [stat['mean'] for stat in stats]
            
    except Exception as e:
        print(f"  -> Failed to process {year}. Error: {e}")
        results_df[f'v_mean_{year}'] = None

# 4. Export
results_df.to_csv(output_csv, index=False)
print(f"\nSuccess! Data exported to {output_csv}")

Loading glacier geometries...

Streaming data for 1990...
  -> Reprojecting shapefile to match remote COG...
  -> Calculating zonal statistics over the cloud...

Streaming data for 2000...
  -> Reprojecting shapefile to match remote COG...
  -> Calculating zonal statistics over the cloud...

Streaming data for 2010...
  -> Reprojecting shapefile to match remote COG...
  -> Calculating zonal statistics over the cloud...

Streaming data for 2015...
  -> Reprojecting shapefile to match remote COG...
  -> Calculating zonal statistics over the cloud...

Streaming data for 2020...
  -> Reprojecting shapefile to match remote COG...
  -> Calculating zonal statistics over the cloud...

Success! Data exported to glacier_velocities_5_epochs.csv


In [8]:
import geopandas as gpd
import xarray as xr
import pandas as pd
from rasterstats import zonal_stats
import os
import rioxarray
import rasterio

# 1. File Paths (Update these to match your files)
shapefile_path = "glaciers_for_gee.shp"
netcdf_path = r"C:\Users\rubel\Downloads\ITS_LIVE_velocity_120m_RGI14A_0000_V02.1.nc"
output_csv = "glacier_velocities_composite1.csv"

cog_urls = r"C:\Users\rubel\Downloads\ITS_LIVE_velocity_120m_RGI14A_0000_V02.1_v.tif"

# 2. Load Glacier Shapefile
print("Loading glacier geometries...")
gdf = gpd.read_file(shapefile_path)

if 'RGIId' not in gdf.columns:
    raise ValueError("The shapefile does not contain an 'RGIId' column!")

results_df = pd.DataFrame({'RGIId': gdf['RGIId']})

# 3. Stream Data from COGs
with rasterio.open(cog_urls) as src:
    cog_crs = src.crs
    
    # Align CRS if they don't match
    if gdf.crs != cog_crs:
        print("  -> Reprojecting shapefile to match remote COG...")
        gdf_projected = gdf.to_crs(cog_crs)
    else:
        gdf_projected = gdf

    print("  -> Calculating zonal statistics over the cloud...")
    # rasterstats natively supports reading directly from remote URLs!
    stats = zonal_stats(
        gdf_projected, 
        cog_urls,               # Point directly to the AWS URL
        stats="mean", 
        nodata=-32767.0
    )
    
    results_df[f'v_mean_{year}'] = [stat['mean'] for stat in stats]
            

# 4. Export
results_df.to_csv(output_csv, index=False)
print(f"\nSuccess! Data exported to {output_csv}")

Loading glacier geometries...
  -> Reprojecting shapefile to match remote COG...
  -> Calculating zonal statistics over the cloud...

Success! Data exported to glacier_velocities_composite1.csv
